## Data Processing in Python Final Project

### Authors: Matyáš Tvrz, Jonathan Eugenio Gaeta

### About:

This is a notebook for the final project for Data Processing in Python. You can either view the project in this notebook, or using the Streamlit environment below, which offers more interactive features.

AI disclaimer: we used Anthropic's Claude for polishing pieces of code for better readability and logic, including docstrings, and for bug fixes.

## View this project in the interactive Streamlit environment!

In [ ]:
# run streamlit environemnt, opens browser
!streamlit run streamlit.py --server.runOnSave true

## Or continue in this notebook...

### Data processing and cleaning

Here, you can update the data using up-to-date information from sreality.cz and bezrealitky.cz. The update should not take longer than 15 minutes. 

ISSUE: sreality dataset cannot be updated, url returns 404 error, most likely sreality removed the API site. update = True then updates only bezrealitky dataset.

In [ ]:
# clean and process data, using update = True requests new data
from src.data_cleaning import process_data
process_data(update = False)

### Heatmap

View the heatmap (choropleth) in your browser:

In [ ]:
# create heatmap (choropleth), opens html in browser
from src.draw_heatmap import draw_heatmap
draw_heatmap()

### Analysis

In [ ]:
# load data for analysis
import src.analysis as an
df_reg = an.load_data()

View summary statistics grouped by flat type:

In [ ]:
# summary statistics by flat type
an.summary_stats_type(df_reg)

We can see that the highest median price per m2 is 1+kk - smaller apartments are relatively more expensive. The overall median rental price is highest for 5+1 and 5+kk apartments, as we would expect.

Or view them grouped by cities/districts (note that what we call districts are either cities which do not have any further sub-regions, or actual districts or streets in cities - the smallest available region for each property):

In [ ]:
# summary statistics by location - filter by city
an.summary_stats_loc_interactive(df_reg)

Prague and its districts dominate in the median price category. Frymburk seems to be also high on the list - perhaps an effect of holiday houses. Regions in the easternmost part of the republic and regions around Ústí nad Labem seem to have the lowest prices.

View exploratory analysis plots:

In [ ]:
# EDA visualizations
an.plot_eda(df_reg)

For big districts, we can see again that Prague has the highest median prices, whereas regions like Ostrava - Poruba or Most have the lowest prices. The violin plot corroborates the earlier finding that 1+kk apartments are most expensive per square meter. Distance to Prague seems to lower the price in a linear way, we can see a spike of price for other big cities as well.

Run level OLS estimation of the equation
$$
\text{Rent}_i=\beta_0+\beta_1\text{Area}_i+\beta_2\text{DistanceToPrague}_i
+\gamma' \text{FlatType}_i+\delta' \text{District}_i+\varepsilon_i.
$$

In [ ]:
ols = an.run_ols(df_reg)
an.plot_diagnostics(df_reg, ols)

Both area and distance to Prague seem to be significant determinants of rental prices, even after accounting for district-specific effects and flat types. Both have the expected sign: area increases price, while distance to Prague decreases it.

Run log-level model:

In [ ]:
# estimate log model
log_ols, baseline_district = an.run_log_ols(df_reg)

The log-level model results corroborate the level model findings, but this model has a better fit. 

View district fixed-effects plot, compared to baseline (closest to median price) district:

In [ ]:
# plot district fixed effect
an.plot_district_fe(log_ols, baseline_district)

Compared to the (median) baseline district (Brno - Bystrc), the monthly rent for properties in the Prague street Veletržní are roughly 175% (coeff. +1.01) higher for an otherwise identical flat. On the other end of the spectrum, rents in Litvínov - Janov are roughly 67% (coeff. -1.1) lower for the same apartment, compared to the baseline. 